# Calculate climate indices

This code calculates the climate indices for the SAI and corresponding SSP2-4.5 experiment. 

You will need:
- daily maximum temperature (TREFHTMX)
- daily minimum temperature (TREFHTMN)
- daily precipitation rate (PRECT)

This code generates the following output (see table below) in the specified folder:
- WARM
- COLD
- WET
- DRY

SOURCE: https://xclim.readthedocs.io/en/stable/indices.html#xclim.indices.prcptot_wetdry_period

| Variable Used | Index | Name | Definition | Unit | Type |
|---|---|---|---|---|---|
| TREFHTMX (Tx) | WARM | Warm days | Number of days when Tx > 25°C | days per month | Fixed threshold |
| TREFHTMN (Tn) | COLD | Cold days | Daily minimum temperature < 0°C | days per month | Fixed threshold |
| PRECT (PR) | DRY | Dry days | Number of days when PR < 1 mm/day | days per month | Fixed threshold |
| PRECT (PR) | WET | Wet days | Number of days when PR > 1 mm/day | days per month | Fixed threshold |


In [1]:
import xarray as xr
import numpy as np
from os.path import join, expanduser
import xclim.indices
import os
import tempfile
from pathlib import Path
import cftime
import shutil


In [ ]:
import dask
# For the SAI experiment or SSP24-5.
num_ensembles = 3 # total number of ensembles

for i in range(1, num_ensembles + 1):
    ensemble_id = f'{(i):03d}'  # 001, 002, 003
    print(f'Loading ensemble member {ensemble_id}')
    file_path = f'/path_to_exp/{ensemble_id}/TREFHTMX_*.nc'  
    tasmax = xr.open_mfdataset(file_path, parallel=True, combine='nested', concat_dim='time')
    tasmax = tasmax.sel(time=~tasmax.get_index("time").duplicated()) # this removes any duplicates
    tasmax = tasmax.isel(time=slice(None, -1))

    file_path = f'/path_to_exp/{ensemble_id}/TREFHTMN_*.nc'  
    tasmin = xr.open_mfdataset(file_path, parallel=True, combine='nested', concat_dim='time')
    tasmin = tasmin.sel(time=~tasmin.get_index("time").duplicated())
    tasmin = tasmin.isel(time=slice(None, -1))

    file_path = f'/path_to_exp/{ensemble_id}/PRECT_*.nc'  
    ds = xr.open_mfdataset(file_path, parallel=True, combine='nested', concat_dim='time')
    ds = ds.sel(time=~ds.get_index("time").duplicated())
    ds = ds.isel(time=slice(None, -1))
    pr = (ds.PRECT*86400*1000).assign_attrs({'units':'mm/d'}) #convert to mm/day here

    # =======================================================
    # Monthly indices
    # =======================================================

    # -- Number of days when Tx > 25 degC
    WARM = xclim.indices.tx_days_above(tasmax.TREFHTMX, thresh='25.0 degC', freq='MS', op='>')

    # -- Monthly count of days when Tn < 0
    COLD = xclim.indices.frost_days(tasmin.TREFHTMN, thresh='0 degC', freq='MS')

    # -- Monthly count of days when PRECT > 10 mm/day
    WET = xclim.indices.wetdays(pr, thresh='10.0 mm/day', freq='MS')

    # -- Monthly count of days when PRECT < 1 mm/day
    DRY = xclim.indices.dry_days(pr, thresh='1 mm/day', freq='MS', op='<')
    
    cold, warm, wet, dry = dask.compute(COLD, WARM, WET, DRY)

    #path to save ETCCDI data for SAI data
    coldnm = join(f'/path_to_save_ETCCDI/', 'COLD.nc')
    warmnm  = join(f'/path_to_save_ETCCDI/', 'WARM.nc')
    wetnm = join(f'/path_to_save_ETCCDI/', 'WET.nc')
    drynm = join(f'/path_to_save_ETCCDI/', 'DRY.nc')

    cold.to_netcdf(coldnm)    
    warm.to_netcdf(warmnm)
    wet.to_netcdf(wetnm)
    dry.to_netcdf(drynm)
    


Loading ensemble member 001


## Double check that the resulting climate indices data variable name reflects the file name. This is to make things easier later

In [4]:
def overwrite_etccdi_varname_inplace(nc_path):
    """
    For an ETCCDI file like WARM.nc, rename the single main data variable
    to match the filename (WARM), and overwrite the file in-place.

    Example:
      WARM.nc contains variable 'TREFHTMX'  -> becomes variable 'WARM'
    """
    nc_path = Path(nc_path)
    new_name = nc_path.stem 

    # Open
    ds = xr.open_dataset(nc_path)

    # Identify the main data variable (skip bounds-like variables)
    candidates = [
        v for v in ds.data_vars
        if ("bnds" not in v.lower()) and ("bounds" not in v.lower())
    ]

    if len(candidates) != 1:
        raise ValueError(
            f"{nc_path} has {len(candidates)} candidate data variables: {candidates}. "
            "Not safe to rename automatically."
        )

    old_name = candidates[0]
    if old_name == new_name:
        print(f"[SKIP] {nc_path.name}: already named '{new_name}'")
        ds.close()
        return

    # Rename
    ds2 = ds.rename({old_name: new_name})
    ds.close()

    # Write to a temporary file in the same directory
    tmp_path = nc_path.with_suffix(".tmp.nc")

    #preserve encoding if present
    encoding = {}
    if old_name in ds2.variables and hasattr(ds2[ new_name ], "encoding"):
        encoding = {new_name: ds2[new_name].encoding}

    ds2.to_netcdf(tmp_path)
    ds2.close()

    # Automatically replace original
    os.replace(tmp_path, nc_path)

    print(f"[OK] {nc_path.name}: '{old_name}' -> '{new_name}' (overwritten)")

In [ ]:
num_ensembles = 3 # total number of ensembles
var_list = 'DRY WET WARM COLD'.split()
for var in var_list:
    for i in range(1, num_ensembles + 1):
        ensemble_id = f'{i:03d}'
        print(f'{var} = Loading ensemble member {ensemble_id}')
        overwrite_etccdi_varname_inplace(f"/path_to_ETCCDI/{ensemble_id}/{var}.nc")
